# 🚀 Модуль 12. Урок 4. Оценка и мониторинг агентных систем в продакшене (Observability & Tracing)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_12_ai_agents/lesson_04_llm_evaluation_and_monitoring.ipynb)

## 📌 Бизнес-контекст и методология
В агентных системах критически важна трассировка (Tracing): сколько шагов сделал агент, какие инструменты вызвал, какова латентность каждого шага и суммарная стоимость сессии.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
# Моделирование объекта трассировки (Trace Log)
trace_log = {
    "trace_id": "tr-9812-agent",
    "total_steps": 3,
    "tools_called": ["sql_runner", "python_evaluator"],
    "total_latency_sec": 2.45,
    "total_cost_usd": 0.0034,
    "status": "SUCCESS"
}

def validate_agent_execution(trace):
    if trace["total_steps"] > 5:
        return False, "Слишком много итераций"
    if trace["total_latency_sec"] > 10.0:
        return False, "Превышен SLA по времени"
    return True, "SLA соблюден"

ok, msg = validate_agent_execution(trace_log)
print(f"Статус проверки трассировки: {ok} ({msg})")


Статус проверки трассировки: True (SLA соблюден)


### 🎯 Практическое задание 1: Проверка SLA
Убедитесь, что тестовый лог удовлетворяет критериям SLA.

In [3]:
### ВАШ КОД ЗДЕСЬ
assert ok == True
print("✅ Тест успешно пройден: мониторинг агентных систем настроен!")


✅ Тест успешно пройден: мониторинг агентных систем настроен!


### 💼 OpenTelemetry в аналитике
Современные платформы (LangSmith, Phoenix, OpenLLMetry) сохраняют полные деревья трассировок вызовов в OpenTelemetry формате.